# Comparativa de ASR usando BERTScore

Este notebook calcula el BERTScore entre las transcripciones normalizadas (`text_normalized`) y las oraciones de referencia (ground truth).

Se utiliza una versión ligera del modelo para una ejecución rápida, pero se deja comentada la opción de usar un modelo más pesado (transformers grandes) para obtener métricas potencialmente más precisas.

In [14]:
# Instalar dependencias si no están presentes
!pip install bert-score transformers torch pandas numpy protobuf sentencepiece huggingface_hub

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import pandas as pd
import json
from bert_score import score
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


## 1. Cargar Datos

In [17]:
# Cargar el dataset normalizado
df = pd.read_csv('/content/drive/MyDrive/normalized_dataset.csv')

# Cargar el ground truth
with open('/content/drive/MyDrive/ground_truth.json', 'r') as f:
    ground_truth = json.load(f)

# Crear un diccionario para mapear id -> texto de referencia
ref_dict = {item['id']: item['text'] for item in ground_truth}

# Verificar las primeras filas
df.head()

,person,audio,noise,snr,provider,text,status,transcription_time,text_normalized
0,p7,1,cafe,0dB,custom,Genera una cotización para el cliente con fáci...,success,1.65,genera 1 cotización para el cliente con fácil ...
1,p7,1,cafe,5dB,custom,Genera una cotización para el cliente con PUC ...,success,1.56,genera 1 cotización para el cliente con puc fa...
2,p7,1,cafe,10dB,custom,Genera una cotización para el cliente con Puff...,success,1.59,genera 1 cotización para el cliente con puffaz...
3,p7,1,clean,clean,custom,Genera una cotización para el cliente con FooF...,success,1.66,genera 1 cotización para el cliente con foofac...
4,p7,1,traffic,0dB,custom,"genera una cotización para el cliente fácil, c...",success,1.53,genera 1 cotización para el cliente fácil con ...


## 2. Preparar Candidatos y Referencias

In [18]:
# Asegurarse de que la columna 'audio' sea string para hacer el mapeo con el id del json
df['audio'] = df['audio'].astype(str)

# Obtener la lista de candidatos (transcripciones normalizadas)
# Rellenar NaNs con string vacío por si acaso
cands = df['text_normalized'].fillna('').tolist()

# Obtener la lista de referencias correspondientes usando la columna 'audio' (que es el ID)
refs = [ref_dict.get(audio_id, "") for audio_id in df['audio']]

# Verificar que tienen la misma longitud
assert len(cands) == len(refs), "Error: La longitud de candidatos y referencias no coincide."

print(f"Total de pares a evaluar: {len(cands)}")
print(f"Ejemplo candidato: {cands[0]}")
print(f"Ejemplo referencia: {refs[0]}")

Total de pares a evaluar: 6000
Ejemplo candidato: genera 1 cotización para el cliente con fácil con 5 monitores led y 3 soportes de pared
Ejemplo referencia: genera 1 cotización para el cliente compufacil con 5 monitores led y 3 soportes de pared


## 3. Calcular BERTScore

Aquí seleccionamos el modelo. 
*   **Modelo ligero (activo):** `distilbert-base-multilingual-cased` (buen balance velocidad/rendimiento).
*   **Modelo medio (comentado):** `PlanTL-GOB-ES/roberta-base-bne` (mejor para español, más lento que distilbert pero razonable).
*   **Modelo pesado (comentado):** `xlm-roberta-large` (mejor rendimiento, mucho más lento y pesado).

In [20]:
# Calcular BERTScore con MODELO ESTÁNDAR
# Ante los errores persistentes con el modelo PlanTL (KeyError, ValueError, OSError, OverflowError),
# optamos por un modelo estándar multilingüe altamente robusto y soportado nativamente.

# Opción 1: DistilBERT (Ligero y rápido) - Descomentar si se quiere velocidad máxima
# model_type = "distilbert-base-multilingual-cased"

# Opción 2: XLM-RoBERTa Base (Balanceado y robusto) - Recomendado ahora
# model_type = "xlm-roberta-base"

# Opción 3: RoBERTa Large BNE (PlanTL) - Solicitado para GPU/Colab
model_type = "xlm-roberta-large"

print(f"Usando modelo robusto: {model_type} en {device}")

# Eliminamos cualquier parche o configuración extraña. BertScore soporta xlm-roberta nativamente.
# Solo especificamos el modelo y lang="es" para que use las capas correctas por defecto.
P, R, F1 = score(
    cands, 
    refs, 
    verbose=True, 
    model_type=model_type, 
    num_layers=17,    # <--- ESTO SOLUCIONA TU ERROR
    device=device
)

# Agregar los resultados al DataFrame
df['bertscore_precision'] = P.cpu().numpy()
df['bertscore_recall'] = R.cpu().numpy()
df['bertscore_f1'] = F1.cpu().numpy()

Usando modelo robusto: xlm-roberta-large en cuda


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/22 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/94 [00:00<?, ?it/s]

done in 6.21 seconds, 965.58 sentences/sec


## 4. Analizar Resultados

In [21]:
# Mostrar estadísticas descriptivas de los scores
print("Estadísticas de BERTScore F1:")
print(df['bertscore_f1'].describe())

# Mostrar promedio agrupado por proveedor (provider)
if 'provider' in df.columns:
    print("\nPromedio de BERTScore F1 por proveedor:")
    print(df.groupby('provider')['bertscore_f1'].mean().sort_values(ascending=False))

# Mostrar promedio agrupado por nivel de ruido (noise)
if 'noise' in df.columns:
    print("\nPromedio de BERTScore F1 por tipo de ruido:")
    print(df.groupby('noise')['bertscore_f1'].mean().sort_values(ascending=False))

# Mostrar promedio agrupado por nivel de ruido (noise)
if 'snr' in df.columns:
    print("\nPromedio de BERTScore F1 por nivel de ruido:")
    print(df.groupby('snr')['bertscore_f1'].mean().sort_values(ascending=False))

Estadísticas de BERTScore F1:
count    6000.000000
mean        0.985814
std         0.023280
min         0.784229
25%         0.977747
50%         1.000000
75%         1.000000
max         1.000000
Name: bertscore_f1, dtype: float64

Promedio de BERTScore F1 por proveedor:
provider
google    0.990411
custom    0.986710
amazon    0.983433
azure     0.982703
Name: bertscore_f1, dtype: float32

Promedio de BERTScore F1 por tipo de ruido:
noise
clean        0.993786
cafe         0.986942
traffic      0.986479
warehouse    0.981364
Name: bertscore_f1, dtype: float32

Promedio de BERTScore F1 por nivel de ruido:
snr
clean    0.990928
10dB     0.987004
5dB      0.977615
0dB      0.954599
Name: bertscore_f1, dtype: float32


In [24]:
# Guardar el dataframe con los scores si es necesario
# df.to_csv('normalized_dataset_with_bertscore.csv', index=False)
df.head()
# df.to_csv('/content/drive/MyDrive/normalized_dataset.csv', index=False)

,person,audio,noise,snr,provider,text,status,transcription_time,text_normalized,bertscore_precision,bertscore_recall,bertscore_f1
0,p7,1,cafe,0dB,custom,Genera una cotización para el cliente con fáci...,success,1.65,genera 1 cotización para el cliente con fácil ...,0.979358,0.963439,0.971333
1,p7,1,cafe,5dB,custom,Genera una cotización para el cliente con PUC ...,success,1.56,genera 1 cotización para el cliente con puc fa...,0.972452,0.963311,0.967860
2,p7,1,cafe,10dB,custom,Genera una cotización para el cliente con Puff...,success,1.59,genera 1 cotización para el cliente con puffaz...,0.963321,0.964065,0.963693
3,p7,1,clean,clean,custom,Genera una cotización para el cliente con FooF...,success,1.66,genera 1 cotización para el cliente con foofac...,0.964208,0.966590,0.965398
4,p7,1,traffic,0dB,custom,"genera una cotización para el cliente fácil, c...",success,1.53,genera 1 cotización para el cliente fácil con ...,0.922836,0.915705,0.919257
